# Lesson 1

Set up

In [ ]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field


In [5]:
# set up OpenAI connection
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [6]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

When querying with more complex prompts (eg asking chatgpt to translate an email), it's good to set up a prompt template.

## Why do we use prompt templates?

Prompts can be long and detailed so prompt templates are a useful abstraction to help you reuse good prompts when you can. LangChain also provides prompts for common operations.

LangChain supports output parsing with prompt templates. You often ask LLM's to generate an output in a certain format. LangChain library functions parse the LLM's output assuming that it will use certain keywords. An example may use Thought, Action, and Observation as keywords for the output to follow a Chain-of-Thought-Reasoning (ReAct) format.

The following code is how you use LangChain to call chatgpt and get the response to a prompt:

In [11]:
chat = ChatOpenAI(temperature = 0.0, model = llm_model)
chat

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x112315a90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x112316510>, root_client=<openai.OpenAI object at 0x1118eb390>, root_async_client=<openai.AsyncOpenAI object at 0x112315be0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

The above line creates a ChatOpenAI instance from LangChain that acts as a wrapper around OpenAI's chat completion API. 

Parameters:
* temperature=0.0: Sets the temperature to 0, which makes the model's responses completely deterministic (no randomness). With temperature 0, the model will always give the same response for the same input prompt.
* model=llm_model: Uses the model specified by the llm_model variable, which in your notebook is set to either "gpt-3.5-turbo" or "gpt-3.5-turbo-0301" depending on the current date.

What it creates:
The chat object is a LangChain ChatOpenAI instance that provides a standardized interface to interact with OpenAI's chat models. It handles:
* API calls to OpenAI
* Message formatting
* Response processing
* Error handling

In [12]:
# creating prompt template
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""


In [13]:
#  creating prompt template using langchain chat prompt template
prompt_template = ChatPromptTemplate.from_template(template_string)

# you can extract the original prompt from the prompt template
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [15]:
# settting the input variables for prompt template

customer_style = """American English \
in a calm and respectful tone
"""

customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [16]:
# generating the prompt

customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [22]:
# you can use the code below to see the type of the object and the content of the prompt
print(type(customer_messages))
print(type(customer_messages[0]))
print(customer_messages[0])
print(customer_messages[0].content) # returns only the content of the prompt

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>
content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}
Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone
. text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [19]:
# Call the LLM using the prompt, returns response which in this case is the tranlated customer message
customer_response = chat(customer_messages)
print(customer_response.content) # this gives you the translated text

Oh man, I'm really frustrated that my blender lid flew off and made a mess of my kitchen walls with smoothie! And on top of that, the warranty doesn't cover the cost of cleaning up my kitchen. I could really use your help right now, buddy.


Output parsers

Output parsers allow you to do something meaningful and useful with the response

In [24]:
# define the template
review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [25]:
# below is the customer review (input)
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

In [26]:
# create the prompt template using the template we created in the last cell
prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [30]:
# create the message to parse to the openai end point
messages = prompt_template.format_messages(text=customer_review)
# create the openai endpoint
chat = ChatOpenAI(temperature=0.0, model=llm_model)
# call the openai endpoint
response = chat(messages)

print(response.content)

# note that when we print the type of the response we see that the response is a string even though it looks like a json/dictionary
type(response.content)


{
    "gift": true,
    "delivery_days": 2,
    "price_value": "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
}


str

In [31]:
# Trying to get the value of the gift
# You will get an error by running this line of code 
# because'gift' is not a dictionary
# 'gift' is a string
response.content.get('gift')

AttributeError: 'str' object has no attribute 'get'

Parse the LLM output string into a Python dictionary

The latest version of LangChain uses pydantic models. A Pydantic model is a Python class that defines the structure and validation rules for data using the Pydantic library. It's like a blueprint for data with built-in validation, type checking, and serialization.

What Pydantic Does
Pydantic is a data validation library that uses Python type annotations to:
* Validate data - ensures data matches expected types and constraints
* Parse data - converts strings/JSON into Python objects
* Serialize data - converts Python objects back to JSON/dicts
* Generate schemas - creates JSON schemas for API documentation

In [35]:
# start by defining what we want the LLM output to look like
# example: extracting information from a product review

class ProductReview(BaseModel):
    gift: bool = Field(description="Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.")
    delivery_days: int = Field(description="How many days did it take for the product to arrive? If this information is not found, output -1.")
    price_value: str = Field(description="Extract any sentences about the value or price, and output them as a comma separated Python list.")

# Create JsonOutputParser with the Pydantic model
output_parser = JsonOutputParser(pydantic_object=ProductReview)

In [36]:
# Using the JsonOutputParser defined above with the Pydantic model
# Get and print the format instructions for the prompt 
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"gift": {"description": "Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.", "title": "Gift", "type": "boolean"}, "delivery_days": {"description": "How many days did it take for the product to arrive? If this information is not found, output -1.", "title": "Delivery Days", "type": "integer"}, "price_value": {"description": "Extract any sentences about the value or price, and output them as a comma separated Python list.", "title": "Price Value", "type": "string"}}, "required": ["gif

In [ ]:
# create prompt template
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

# create prompt template using template
prompt = ChatPromptTemplate.from_template(template=review_template_2)

# creating prompt
messages = prompt.format_messages(
    text=customer_review,
    format_instructions=format_instructions
)

# have a look at the content of the prompt
print(messages[0].content)


response = chat(messages)
print(response.content)
output_dict = output_parser.parse(response.content)
output_dict
type(output_dict)
output_dict.get('delivery_days')



For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties

2

In [39]:
# call chatgpt with prompt
response = chat.invoke(messages)  # prefer .invoke over __call__
print('This is the content of the response variable:')
print(response.content, '\n')

# Parse into a validated Python object (Pydantic model)
parsed = output_parser.parse(response.content) # it's good to paerse into a dict because then you can extract values in the dict

print('This is the content of the parsed output:')
print(parsed, '\n')
print(type(parsed))  

# validate into your pydantic model to then be able to use it's attributes
parsed = ProductReview.model_validate(parsed)  
print(parsed.delivery_days, '\n')         # access as attribute
print(parsed.model_dump())        # how to access all the parsed data as a dictionary

This is the content of the response variable:
{
  "gift": false,
  "delivery_days": 2,
  "price_value": "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
} 

This is the content of the parsed output:
{'gift': False, 'delivery_days': 2, 'price_value': "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."} 

<class 'dict'>
2 

{'gift': False, 'delivery_days': 2, 'price_value': "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."}
